# CineScope — Baseline Validation

Reads the bronze `movies_ratings` Parquet table from the external SSD. Does **not** re-run raw ETL.

In [ ]:
from pathlib import Path
import sys

from dotenv import load_dotenv

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
load_dotenv(REPO / ".env")

from cinescope.paths import get_paths
from cinescope.spark_session import build_spark_session

paths = get_paths(create_dirs=False, validate_mount=True)
spark = build_spark_session(app_name="cinescope-baseline-notebook", paths=paths)
print("SSD data root:", paths.data_root)
print("Parquet:", paths.movies_ratings_dir)

In [ ]:
df = spark.read.parquet(str(paths.movies_ratings_dir))
df.printSchema()
row_count = df.count()
print("row_count:", row_count)
df.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

df.select(
    F.count("*").alias("n"),
    F.avg("average_rating").alias("avg_rating"),
    F.avg("num_votes").alias("avg_votes"),
    F.avg("runtime_minutes").alias("avg_runtime"),
    F.min("start_year").alias("min_year"),
    F.max("start_year").alias("max_year"),
).show()

# Interpretation: aggregate stats stay in Spark; no full collect to pandas.

In [ ]:
null_exprs = [
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]
df.agg(*null_exprs).show()

# Interpretation: nulls in runtime/year/genres are expected from IMDb \\N sentinels.

In [ ]:
top_genres = (
    df.select(F.explode_outer("genres").alias("genre"))
    .filter(F.col("genre").isNotNull())
    .groupBy("genre")
    .count()
    .orderBy(F.desc("count"))
)
top_genres.show(15)

# Interpretation: temporary explode for genre frequency only; bronze table keeps genres as arrays.

In [ ]:
by_decade = (
    df.filter(F.col("start_year").isNotNull())
    .withColumn("decade", (F.floor(F.col("start_year") / 10) * 10).cast("int"))
    .groupBy("decade")
    .agg(
        F.count("*").alias("movies"),
        F.avg("average_rating").alias("avg_rating"),
        F.avg("num_votes").alias("avg_votes"),
    )
    .orderBy("decade")
)
by_decade.show(50)

# Interpretation: decade aggregates show volume growth and rating/vote shifts over time.

In [ ]:
df.filter(F.col("runtime_minutes").isNotNull()).select(
    F.min("runtime_minutes").alias("min_runtime"),
    F.expr("percentile_approx(runtime_minutes, 0.5)").alias("p50_runtime"),
    F.expr("percentile_approx(runtime_minutes, 0.9)").alias("p90_runtime"),
    F.max("runtime_minutes").alias("max_runtime"),
    F.avg("runtime_minutes").alias("avg_runtime"),
).show()

# Interpretation: runtime distribution helps spot outliers before feature engineering.
spark.stop()